# 11 · Datos del SENAMHI: ¿sirven los pluviómetros reales?

> **No es un sistema de alerta ni una predicción para decidir evacuaciones.** Ante una emergencia llama al 115 (INDECI).

**Fuente:** Información recopilada y trabajada por el Servicio Nacional de Meteorología e Hidrología del Perú. El uso que se le da a esta información es de mi (nuestra) entera responsabilidad. Fuente: SENAMHI, [Observatorio Nacional del Tiempo, Agua y Clima del Perú](https://observatorio.senamhi.gob.pe).

Los [términos de uso](https://www.senamhi.gob.pe/?p=terminos-condiciones) permiten usar la información, con o sin fines de lucro, reconociendo al SENAMHI con esa leyenda en todo soporte; no se puede comercializar ni presentar como oficial del SENAMHI, del MINAM o del Estado. El portal avisa que **los datos no tienen control de calidad**. Las series se guardan en `datos/senamhi/` (fuera de git) y viajan al espejo de Hugging Face con la leyenda y estas condiciones en su tarjeta.

Para correrlo: `uv run descargar-datos --solo senamhi` (se pide aparte por tamaño y cortesía: mes a mes, con 2 s de pausa) o `uv run --extra hf descargar-datos --desde-hf USUARIO/REPO`.

**Teorías (escritas antes de mirar):**
- **T30** los pluviómetros de la cuenca media-alta (Autisha, Matucana, Santiago de Tuna...) detectan los días con huaico mejor que ERA5, porque ERA5 promedia 25 km y el huaico depende de tormentas locales;
- **T31** el pluviómetro del propio Chosica casi no mide la lluvia que activa las quebradas (cae arriba, no en el pueblo), pero que llueva algo en el pueblo ya separa los días con huaico;
- **T32** el nivel del río en Chosica concuerda solo en parte con el caudal de GloFAS;
- **T33** las reglas oficiales de umbral de lluvia del SENAMHI (`g_umbrales`) tienen un POD parecido al de nuestros modelos al mismo número de días de alarma, con tanta alarma falsa como ellos;
- **T34** los avisos meteorológicos históricos del GeoServer (solo 2020) marcan los huaicos de 2020 pero con casi todos los días de lluvia como aviso;
- **T35** la anomalía de lluvia de los Niños del SENAMHI (1983, 1998, 2017, 2023, 2024) ordena los trimestres por días de huaico.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

from illapa_research import estudio_eventos as ee
from illapa_research import evaluacion_diaria as E
from illapa_research import predictores as P
from illapa_research import senamhi as S
from illapa_research.catalogo import cargar_todos

RAIZ = Path("..") if Path("../datos").exists() else Path(".")
CARPETA = RAIZ / "datos" / "senamhi"
HAY_DATOS = (CARPETA / "cache").exists() and any((CARPETA / "cache").glob("researchers__*"))
print("datos locales del SENAMHI:", "sí" if HAY_DATOS else "no (corre `descargar-datos --solo senamhi`)")
print(S.CITA)

## 1. Qué estaciones hay cerca de Chosica

Metadatos del catálogo público (estos sí se pueden publicar): `catalogo/senamhi_estaciones_cercanas.csv`, las estaciones meteorológicas e hidrológicas a menos de 45 km, con los meses de datos que publica el servicio.

In [ ]:
inv = pd.read_csv(RAIZ / "catalogo" / "senamhi_estaciones_cercanas.csv")
print(inv.groupby(["tipo", "condicion"]).size().unstack(fill_value=0))
inv[inv.meses_con_dato > 0].sort_values("km_a_chosica")[["nombre", "tipo", "categoria", "km_a_chosica", "altitud_m", "desde", "hasta", "meses_con_dato"]].head(25)

**Qué lectura da el servicio.** Hay dos puertas, ambas anónimas y las dos las usa la propia página «Datos hidrometeorológicos»: una serie **diaria histórica** por estación (precipitación y temperaturas extremas, de 1963 o 1989 hasta enero de 2014) y la **lectura mensual** de las estaciones vigentes (de 2013 o 2018 en adelante; las automáticas, por hora). Entre 2014 y 2017 no hay pluviómetro convencional publicado en Chosica; la estación hidrometeorológica automática de Chosica, con lluvia y nivel del río por hora, empieza en enero de 2001.

In [ ]:
if HAY_DATOS:
    c = S.Cliente(CARPETA)
    hist = S.red_historica(c)
    auto = S.red_automatica(c)
    cobertura = pd.concat({"convencionales": hist.notna().resample("YS").mean(), "automáticas": auto.notna().resample("YS").mean()}, axis=1)
    display((cobertura.loc["1989":] * 100).round(0).iloc[::2])

## 2. ¿Cuánto concuerda ERA5 con un pluviómetro real?

In [ ]:
if HAY_DATOS:
    ev = [pd.Timestamp(e.fecha) for e in cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True)]
    pred, _ = P.cargar_con_respaldo(RAIZ / "datos", rellenar_con_power=False)
    horario = ee.cargar_horario(RAIZ / "datos" / "open_meteo_horario_estudio.json")
    era_chosica = horario["chosica"]["precipitation"].resample("D").sum()
    filas = []
    for nombre, g, e_ in (
        ("Matucana (convencional) vs ERA5, 1989-2013", hist["matucana"]["1989":"2013"], pred["matucana"]["precipitation_sum"]),
        ("Matucana (automática) vs ERA5, 2013-2026", auto["matucana_auto"], pred["matucana"]["precipitation_sum"]),
        ("Chosica (automática) vs ERA5, 2001-2026", auto["chosica_auto"], era_chosica),
    ):
        a = S.acuerdo(g, e_, ev, 0)
        filas.append({"par": nombre, **{k: round(v, 2) for k, v in a.items()}})
    display(pd.DataFrame(filas).set_index("par"))

Lectura: ERA5 y el pluviómetro se parecen poco en el día a día (Spearman 0,4 a 0,6) y ERA5 da más lluvia en los extremos (p95 de 18 mm contra 7 mm en Matucana). **En Chosica, los días con huaico el pluviómetro del pueblo mide poco (mediana 0,9 mm) mientras ERA5 dice 5 mm. Aun así separa bien los días (AUC 0,80 la automática, 0,67 la convencional): T31 se cumple a medias.**

## 3. ¿Qué estación discrimina los días con huaico?

AUC de la lluvia del mismo día de cada estación (0,5 = nada), solo en temporada de lluvias.

In [ ]:
if HAY_DATOS:
    idx = pd.date_range("1984-01-01", "2026-10-29", freq="D")
    y = E.etiquetas(cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True), idx)
    m = P.temporada_mask(idx)
    filas = []
    for nombre, R, a, b in (("convencionales 1990-2013", hist, 1990, 2013), ("automáticas 2014-2026", auto, 2014, 2026)):
        mm = m & (idx.year >= a) & (idx.year <= b)
        for col in R.columns:
            x = R[col].reindex(idx)[mm]
            ok = x.notna()
            if y[mm][ok].sum() >= 5 and ok.mean() > 0.3:
                filas.append({"red": nombre, "estacion": col, "auc": roc_auc_score(y[mm][ok], x[ok]), "dias_con_dato": int(ok.sum())})
    display(pd.DataFrame(filas).sort_values(["red", "auc"], ascending=[True, False]).round(3))

## 4. ¿Mejora el modelo diario del capítulo 05?

Se repite el modelo (regresión logística con regularización elegida dentro del entrenamiento, dejando una temporada fuera) con tres juegos de lluvia: ERA5, pluviómetros, y ambos. Los pluviómetros dan la lluvia **del día**, así que es un *nowcast* (el valor diario se lee a la mañana siguiente), no un pronóstico.

In [ ]:
if HAY_DATOS:
    r1 = S.experimento_pluviometros(RAIZ, hist, 1990, 2013)
    print(r1["n_dias"], "días,", r1["n_eventos"], "con huaico,", r1["n_temporadas"], "temporadas")
    display(r1["tabla"][["ap", "ap_base", "auc", "brier_skill", "aciertos_con_5pc_alarmas", "aciertos_con_10pc_alarmas"]].round(3))
    print("pluviómetros vs ERA5 (AP):", {k: round(v, 3) for k, v in S.diferencia_ap(r1["y"], r1["p"]["pluviómetros del día"], r1["p"]["ERA5 lluvia del día"], r1["ts"]).items()})

In [ ]:
if HAY_DATOS:
    red_auto = auto[[c for c in auto.columns if c not in ("nana_auto",)]]
    r2 = S.experimento_pluviometros(RAIZ, red_auto, 2014, 2026, separar_valle=False)
    print(r2["n_dias"], "días,", r2["n_eventos"], "con huaico,", r2["n_temporadas"], "temporadas; estaciones:", list(red_auto.columns))
    display(r2["tabla"][["ap", "ap_base", "auc", "brier_skill", "aciertos_con_5pc_alarmas", "aciertos_con_10pc_alarmas"]].round(3))
    for a, b in (("pluviómetros del día", "ERA5 lluvia del día"), ("ERA5 día + antecedentes + pluviómetros", "ERA5 día + antecedentes")):
        print(a, "vs", b, {k: round(v, 3) for k, v in S.diferencia_ap(r2["y"], r2["p"][a], r2["p"][b], r2["ts"]).items()})

## 5. Nivel del río en Chosica contra el caudal de GloFAS

In [ ]:
if HAY_DATOS:
    from scipy.stats import spearmanr

    ch = c.serie("47278214")
    nivel = pd.to_numeric(ch["N_NIVELAGUA"], errors="coerce")
    n_h = nivel.resample("D").count()
    nivel_d = nivel.resample("D").mean().where(n_h >= 20)
    glofas = np.log(P.cargar_caudal(RAIZ / "datos" / "open_meteo_caudal_glofas.json")["rimac-chosica"])
    j = pd.concat([nivel_d.rename("nivel_m"), glofas.rename("log_caudal_glofas")], axis=1).dropna().loc["2001":]
    print(len(j), "días; Spearman nivel (m) vs log caudal GloFAS:", round(spearmanr(j.nivel_m, j.log_caudal_glofas)[0], 2))

## 6. Umbrales oficiales de lluvia (`g_umbrales:umbrales_precipitacion`)

Capa pública con 215 estaciones: lluvia diaria (`umbral`) y acumulada (`umb_acum`) que activan la alerta. En las estaciones del Rímac: Chosica y Ñaña 1 mm / 3 mm, Santa Eulalia 2 / 6, Matucana y San Mateo de Otao 4 / 12. La ventana de la acumulada no viene en la capa: se leyó como 3 días (es 3 veces el umbral en todas). Regla evaluada: *alarma el día en que la lluvia del día llega al umbral o la de los últimos 3 días a la acumulada*.

In [ ]:
if HAY_DATOS:
    um = S.umbrales(c)
    zona = um[um["nombre"].isin(["CHOSICA", "ÑAÑA", "MATUCANA", "SANTA EULALIA", "SAN MATEO DE OTAO", "CASAPALCA"])]
    display(zona[["nombre", "distrito", "cuenca", "umbral", "umb_acum"]].reset_index(drop=True))

In [ ]:
if HAY_DATOS:
    idx_d = pd.date_range("1984-01-01", "2026-10-29", freq="D")
    y_d = E.etiquetas(cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True), idx_d)
    temporada = P.temporada_mask(idx_d)
    umb = {r.nombre: (r.umbral, r.umb_acum) for r in um.itertuples()}
    filas = []
    for nombre, red, a, b in (("convencionales 1990-2013", hist, 1990, 2013), ("automáticas 2014-2026", auto, 2014, 2026)):
        m_ = temporada & (idx_d.year >= a) & (idx_d.year <= b)
        alarmas = {}
        for k in red.columns:
            if k in S.UMBRAL_DE and S.UMBRAL_DE[k] in umb:
                alarmas[k] = S.alarma_por_umbral(red[k], *umb[S.UMBRAL_DE[k]], idx_d)
                filas.append({"red": nombre, "regla": k, **S.evalua_alarmas(alarmas[k], y_d, m_)})
        cuantas = pd.concat(alarmas.values(), axis=1)
        cualquiera = cuantas.max(axis=1).where(cuantas.notna().sum(axis=1) >= 2)
        filas.append({"red": nombre, "regla": "cualquier estación", **S.evalua_alarmas(cualquiera, y_d, m_)})
    display(pd.DataFrame(filas).round(3))

Para comparar con los modelos al mismo número de días de alarma: POD y FAR de cada modelo del §4 cuando alarma el mismo porcentaje de días que la regla «cualquier estación».

In [ ]:
if HAY_DATOS:
    filas = []
    for r_, frac in ((r1, 0.271), (r2, 0.361)):
        for k, p in r_["p"].items():
            if k != "solo fecha":
                filas.append({"ventana": f"{r_['fechas'][0].year}-{r_['fechas'][-1].year}", "modelo": k, "% de días en alarma": frac, **S.probabilidad_con_presupuesto(p, r_["y"], frac)})
    display(pd.DataFrame(filas).round(2))

## 7. Avisos meteorológicos históricos (`g_100_01:aviso001…aviso273`, solo 2020)

Cada capa `avisoNNN` es un aviso numerado de 2020 (el `aviso001` se emitió el 1 de enero de 2020 y el `aviso270` el 29 de diciembre de 2020; del 272 en adelante hay filas vacías). No hay avisos de 2021 a hoy en el GeoServer (solo el vigente, `g_aviso:view_aviso`). Se tomaron los que contienen el punto de Chosica (o de Matucana), con el nivel y el fenómeno de lluvia en sierra o costa.

In [ ]:
AV = CARPETA / "avisos_2020_puntos.json"
if HAY_DATOS and AV.exists():
    import json

    av_p = pd.DataFrame(json.loads(AV.read_text(encoding="utf-8")))
    for col in ("fech_ini", "fech_fin"):
        av_p[col] = pd.to_datetime(av_p[col], format="%d_%m_%Y", errors="coerce")
    av_p["nivel_n"] = av_p["nivel"].str.extract(r"(\d)").astype(float)
    av_p = av_p.dropna(subset=["fech_ini", "fech_fin"])
    dias2020 = pd.date_range("2020-01-01", "2020-12-31")
    huaicos2020 = {pd.Timestamp(e.fecha) for e in cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True) if e.fecha.year == 2020}
    filas = []
    for punto in ("Chosica", "Matucana"):
        sd = S.serie_diaria_avisos(av_p[av_p["punto"] == punto], dias2020)
        temporada_2020 = dias2020[(dias2020.month <= 4) | (dias2020.month == 12)]
        n = sd["nivel_max"].reindex(temporada_2020)
        yy = pd.Series([int(d in huaicos2020) for d in temporada_2020], index=temporada_2020)
        yt = (yy | yy.shift(1, fill_value=0) | yy.shift(-1, fill_value=0)).astype(int)
        for lim in (1, 2, 3):
            a = n >= lim
            filas.append({"punto": punto, "aviso de nivel >=": lim, "días de aviso (temporada)": int(a.sum()), "% de días": round(a.mean(), 2),
                          "huaicos con aviso": f"{int(yy[a].sum())}/{int(yy.sum())}", "FAR_días": round(float((yt[a] == 0).mean()), 2) if a.sum() else None})
    display(pd.DataFrame(filas))

## 8. Anomalía de lluvia de los Niños (`g_02_01…g_02_06`)

Clases de anomalía por trimestre (DEF, EFM, FMA) de los eventos de 1983, 1998, 2017, 2023 y 2024; la unidad no figura en la capa. Se leyó la clase que contiene cada punto de estudio (consulta WFS con `INTERSECTS`; ojo: en WFS 2.0 con EPSG:4326 el orden del punto es *latitud longitud*).

In [ ]:
if HAY_DATOS:
    from scipy.stats import spearmanr

    an = S.anomalias_ninos(c)
    fechas_h = [e.fecha for e in cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True)]
    an["dias_huaico"] = [S.dias_huaico_por_trimestre(fechas_h, a, t) for a, t in zip(an["evento"], an["trimestre"], strict=True)]
    tabla = an.pivot_table(index=["evento", "trimestre"], columns="punto", values="anomalia_centro")
    tabla["dias_huaico"] = an.groupby(["evento", "trimestre"])["dias_huaico"].first()
    display(tabla.round(0))
    for punto in S.PUNTOS_ESTUDIO:
        x = tabla[[punto, "dias_huaico"]].dropna()
        rho, p = spearmanr(x[punto], x["dias_huaico"])
        print(f"{punto:14s} n={len(x):2d}  Spearman anomalía vs días de huaico = {rho:+.2f} (p = {p:.3f})")

## 9. Conjuntos con licencia explícita en datosabiertos.gob.pe (ODC-By)

Dos de los cinco conjuntos del SENAMHI en la Plataforma Nacional de Datos Abiertos tienen licencia *Open Data Commons Attribution*: «Variables meteorológicas de las estaciones automáticas de intercambio internacional» (horario; incluye Matucana, 2015 a junio de 2024) y «Variables ambientales atmosféricas del Observatorio de Marcapomacocha» (diario, 2015 a 2024).

In [ ]:
DA = CARPETA / "datosabiertos"
if (DA / "intercambio.csv").exists():
    i = pd.read_csv(DA / "intercambio.csv", low_memory=False)
    m_ = i[i.ESTACION == "MATUCANA"].copy()
    m_["t"] = pd.to_datetime(m_.FECHA.astype(str), format="%Y%m%d") + pd.to_timedelta(m_.HORA.astype(int) // 10000, unit="h")
    pp = m_.set_index("t")["PP"].astype(float)
    dia = pp.resample("D").sum(min_count=1).where(pp.resample("D").count() >= 20)
    comun = pd.concat([dia.rename("ODC-By"), auto["matucana_auto"].rename("portal")], axis=1).dropna()
    print("Matucana horario ODC-By vs la estación automática del portal:", len(comun), "días comunes; Spearman", round(spearmanr(comun["ODC-By"], comun["portal"])[0], 3))

## Lo que no se puede concluir

- Son pocas estaciones y los registros convencionales terminan en enero de 2014; entre 2014 y 2017 la red de montaña casi no tiene datos públicos.
- No hay control de calidad. Los avisos históricos del GeoServer cubren solo 2020 (3 huaicos catalogados ese año), así que la comparación con avisos tiene muy poca potencia; de 2021 a hoy hace falta la API con cuenta (que no se usa).
- Las series diarias convencionales se leen a las 07:00 y cubren las 24 horas previas; aquí se alinean con el día de calendario del mismo valor, que fue el desfase que más concordó con ERA5.
- La mejora se mide dentro de ventanas cortas (25 y 13 temporadas, 53 y 51 días con huaico): los intervalos son anchos aunque la diferencia es clara.